## Setup

### Imports

In [2]:
import sys
import os 

# necessary to make data loading functions available
# Jupyter server should be running from src/python
sys.path.append(
    os.getcwd()
)

In [3]:
from pilot.data_import import (
    import_acetyl,
    import_global,
    import_lipids,
    import_meta,
    import_metabolites,
    import_phospho,
    import_rna,
    syn_login
)

In [1]:
from collections import defaultdict

import polars as pl
import pandas as pd
import synapseclient as sc
from synapseclient.models import File as scFile

### Synapse login token/session

In [4]:
SYN = syn_login()


UPGRADE AVAILABLE

A more recent version of the Synapse Client (4.12.0) is available. Your version (4.11.0) can be upgraded by typing:
   pip install --upgrade synapseclient

Python Synapse Client version 4.12.0 release notes

https://python-docs.synapse.org/news/


Welcome, dylan.ross!



### Constants

In [16]:
# Jupyter server should be running from src/python
CACHE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "mutations",
    "_cache"
)

# metadata and combined -omics data saved to Apache arrow files
# for quick and easy loading
META_CACHE = os.path.join(CACHE_DIR, "metadata.arrow")
META_CACHE_CSV = os.path.join(CACHE_DIR, "metadata.csv")
COMBINED_CACHE = os.path.join(CACHE_DIR, "combined_omics_data.arrow")
COMBINED_CACHE_CSV = os.path.join(CACHE_DIR, "combined_omics_data.csv")

# folder ID for storing combined data table on synapse
# (syn74722605/Analysis/Exp26 Analysis/mutations/)
SYN_UPLOAD_FOLDER_ID = "syn74775762"

### Ensure cache directory is available

In [17]:
os.makedirs(CACHE_DIR, exist_ok=True)

## Load Metadata and Each -Omics Dataset separately
### Metadata

In [7]:
meta = (
    pl.from_pandas(
        import_meta(SYN), 
        include_index=True
    )
    .rename({"None": "Sample"})
    # keep only a subset of the mutation data:
    # important mutations from Eisfeld paper
    # https://doi.org/10.1038/s41588-024-01929-x
    .select(
        "Sample",
        "Age",
        "Sex",
        "Race",
        "Study", 
        # for all of the mutation data: 
        #   (implicit) treat "Not measured" as "WT"
        #   (explicit) fill any null values with "WT"
        # convert the column into boolean indicating mutation state
        (pl.col("FLT3_ITD").fill_null("WT") == "Mutant"),
        (pl.col("NPM1").fill_null("WT") == "Mutant"),
    )
    # retain only samples with "White" or "Black" race label
    .filter(pl.col("Race").is_in(["White", "Black"]))
)
meta

[syn69692583:updated_meta.csv]: Found existing file at /Users/dylan.ross/.synapseCache/79/170282079/updated_meta.csv, skipping download.
[syn64126463:beataml_waves1to4_sample_mapping.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/872/150013872/beataml_waves1to4_sample_mapping.xlsx, skipping download.
[syn64126458:1-s2.0-S1535610822003129-mmc2.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/845/150013845/1-s2.0-S1535610822003129-mmc2.xlsx, skipping download.
[syn26427388:beataml_wes_wv1to4_mutations_v4.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/369/83911369/beataml_wes_wv1to4_mutations_v4.xlsx, skipping download.
[syn32533104:beataml_wes_wv1to4_mutations_177_samples.txt]: Found existing file at /Users/dylan.ross/.synapseCache/976/98264976/beataml_wes_wv1to4_mutations_177_samples.txt, skipping download.


Sample,Age,Sex,Race,Study,FLT3_ITD,NPM1
str,f64,str,str,str,bool,bool
"""11-00261""",74.0,"""Male""","""White""","""BeatAML""",true,false
"""11-00503""",54.0,"""Female""","""White""","""BeatAML""",true,true
"""11-00475""",65.0,"""Male""","""White""","""BeatAML""",true,true
"""12-00032""",70.0,"""Male""","""White""","""BeatAML""",true,false
"""11-00376""",49.0,"""Male""","""White""","""BeatAML""",true,true
…,…,…,…,…,…,…
"""16-01109-Bridge""",33.0,"""Male""","""Black""","""pilotStudy""",false,false
"""16-01191-Bridge""",65.0,"""Female""","""White""","""pilotStudy""",false,false
"""17-00025-Bridge""",39.0,"""Female""","""White""","""pilotStudy""",true,true


### Acetylomics

In [8]:
_ptrc, _pilot = import_acetyl(SYN)
acetyl = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Acetylomics").alias("Block"),
        pl.exclude("Block")
    )
)
acetyl

[syn53484994:ptrc_ex10_crosstab_acetyl_siteID_corrected_relaxed.txt]: Found existing file at /Users/dylan.ross/.synapseCache/792/133616792/ptrc_ex10_crosstab_acetyl_siteID_corrected_relaxed.txt, skipping download.
[syn69075568:ptrc_ex26_crosstab_acetyl_siteid_corrected.txt]: Found existing file at /Users/dylan.ross/.synapseCache/514/161922514/ptrc_ex26_crosstab_acetyl_siteid_corrected.txt, skipping download.
[syn25807733:Ex10_metadata.txt]: Found existing file at /Users/dylan.ross/.synapseCache/668/78060668/Ex10_metadata.txt, skipping download.
[syn68835814:PTRC_Exp26 Sample Key.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/432/161070432/PTRC_Exp26 Sample Key.xlsx, skipping download.


Block,Feature,12-00032,11-00503,12-00069,13-00468,18-00103,12-00123,18-00101,12-00127,12-00145,12-00154,16-00056,16-00088,16-00094,18-00251,16-00109,16-00120,18-00260,16-00265,12-00196,16-00271,16-00289,16-00292,16-00354,15-00880,16-00306,16-00310,18-00278,16-00459,16-00886,12-00250,16-01237,16-00519,16-00538,16-00611,16-00627,…,C-05-4372,C-07-2070,C-07-2900,C-08-2081,C-08-2480,C-08-3337,C-08-3381,C-08-3493,C-09-0923,C-09-1033,C-09-1074,C-09-1336,C-09-1608,C-09-1906,C-09-2182,C-09-3512,C-09-4769,C-09-5381,C-09-5462,C-10-0302,C-10-0535,C-10-0773,C-10-3265,C-10-3906,C-10-3924,C-11-0287,C-11-2295,C-11-5466,C-12-0858,C-12-1118,C-12-2943,C-12-4258,C-13-0276,16-00627-Bridge,16-01191-Bridge,16-00731-Bridge,14-00528-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Acetylomics""","""ACAA2-K234k""",-0.762,-1.32,1.56,null,0.162,0.484,null,null,-0.946,0.776,-0.979,-0.517,0.42,null,null,-1.19,null,-0.67,0.496,0.396,null,0.325,null,-0.00501,0.0123,-1.43,null,-1.4,-1.03,0.42,null,null,null,-2.09,null,…,-0.108936,0.695295,0.58002,0.826197,1.312417,1.059082,0.236566,-0.258634,-1.021999,1.525606,-2.707088,1.138152,-0.848855,1.3347,-0.480795,0.22329,1.527838,-1.592412,0.637115,0.897605,-1.822705,1.615072,0.615995,-0.733113,1.426896,0.882817,1.689777,1.809463,-0.501932,-0.550935,-0.159147,-1.237468,1.30142,-1.067707,0.360673,-0.778423,-1.686547
"""Acetylomics""","""ACAD9-K202k""",null,-0.357,0.123,null,0.851,null,-0.108,-2.44,0.438,0.645,null,null,-0.717,0.294,0.0,null,-0.5,-1.95,-0.0222,-0.598,-0.288,null,-0.0604,0.915,null,-0.561,-0.412,-0.967,0.287,0.18,1.54,0.219,-0.257,null,0.169,…,1.627,null,1.094688,1.006112,1.288685,0.595068,0.614767,-0.978994,null,0.570992,null,0.385527,1.020326,null,null,null,1.155602,0.775528,null,0.133628,0.460517,null,null,-0.026069,-0.581955,null,1.075163,null,null,null,null,0.076998,null,0.207464,-0.602037,-0.395339,null
"""Acetylomics""","""ACADM-K304k""",null,-0.0232,-0.0467,null,-0.987,0.114,0.887,-0.0785,-1.07,-0.226,-0.0843,-1.64,0.103,0.143,0.95,null,-0.301,-0.599,0.286,-0.869,null,null,null,-0.718,null,null,-1.35,-1.84,-0.119,0.64,1.59,-0.34,-0.718,null,-0.752,…,0.675267,0.632392,0.392967,0.581864,1.077454,0.350619,-0.383106,-1.128983,null,-0.310245,-0.499941,0.20996,0.047455,0.18087,-0.762945,-0.543041,1.162907,-0.161699,-0.146591,0.021127,-0.273264,1.79485,null,-1.030927,-0.025516,null,1.54642,null,null,-0.212199,0.44776,-0.535991,null,-0.672011,-0.994443,-1.563158,null
"""Acetylomics""","""ACADVL-K278k""",null,0.81,-0.415,null,null,1.16,null,0.726,-1.03,0.279,0.0793,-0.0988,-0.0384,0.587,0.554,-1.09,null,-1.04,1.4,1.42,0.737,0.394,null,-0.00633,-0.457,0.528,null,1.27,-0.25,-1.43,0.512,0.406,null,-1.54,-2.4,…,-0.064754,0.249759,0.458229,-0.119824,1.687356,-0.13466,0.40549,-1.207801,-1.592601,1.182928,-1.148734,0.472807,-0.261355,-1.930121,-0.378173,-0.404381,-0.193404,-0.344209,-1.074484,0.312725,-0.523238,1.503545,1.316327,0.048724,0.843859,1.160454,1.280458,0.225797,-0.152875,0.452315,0.15943,-1.552973,-0.033898,-2.109005,-1.049815,-1.389221,0.002312
"""Acetylomics""","""ACADVL-K298k""",-2.19,0.421,0.21,null,null,0.699,0.312,-0.605,0.465,0.128,-1.63,-0.725,null,0.602,0.379,-1.09,-0.578,-0.66,0.915,-0.643,-0.103,-0.928,null,1.03,null,0.18,0.394,-1.47,-0.158,-0.371,0.994,-0.0696,0.198,null,-0.354,…,-0.125828,0.108751,0.528723,-0.140161,0.722645,-0.048628,0.215458,-1.040968,-0.505037,0.883659,-0.792169,1.29731,-0.066309,0.605026,-0.978926,-1.543227,-0.112029,0.017959,null,-0.272754,-0.808564,0.305734,0.771271,-0.888301,-0.473342,0.961534,0.861988,0.783867,-0.746466,0.250597,0.035256,-0.903754,-0.093175,-0.655982,-0.471836,-0.293991,0.303983
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

### Lipidomics

In [9]:
_ptrc, _pilot = import_lipids(SYN)
lipid = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Lipidomics").alias("Block"),
        pl.exclude("Block")
    )
)
lipid

[syn71896673:f_data_lipid.csv]: Found existing file at /Users/dylan.ross/.synapseCache/689/170119689/f_data_lipid.csv, skipping download.
[syn71896667:normalized_combat_edata_lipid.csv]: Found existing file at /Users/dylan.ross/.synapseCache/248/166162248/normalized_combat_edata_lipid.csv, skipping download.


Block,Feature,12-00032,11-00503,12-00069,13-00468,18-00103,12-00123,18-00101,12-00127,12-00145,12-00154,12-00196,12-00250,12-00268,12-00294,18-00149,12-00383,13-00016,13-00033,13-00034,13-00047,13-00059,13-00075,13-00077,13-00092,13-00123,13-00157,13-00160,13-00186,13-00195,13-00262,13-00331,13-00393,13-00149,13-00147,13-00450,…,C-08-2091,C-08-2480,C-08-3337,C-08-3381,C-08-3493,C-09-0143,C-09-0261,C-09-0923,C-09-1033,C-09-1074,C-09-1336,C-09-1608,C-09-1906,C-09-2182,C-09-3512,C-09-4769,C-09-5381,C-09-5462,C-10-0302,C-10-0535,C-10-0773,C-10-1211,C-10-3265,C-10-3429,C-10-3906,C-10-3924,C-11-0287,C-11-2295,C-11-5466,C-12-0858,C-12-1118,C-12-2943,C-12-4258,C-13-0276,16-00627,16-01191,16-00731
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lipidomics""","""neg_LPI 20:4_[M-H]-__A""",18.435897,19.425027,18.781972,19.559651,19.229072,18.448508,18.462212,19.008276,17.816789,17.262339,17.467426,19.47388,18.541375,19.466617,17.972661,17.910608,18.993415,17.06623,19.026417,18.789276,18.182538,18.497899,19.132243,18.712868,18.039241,17.776844,18.105979,17.44444,18.57031,19.164063,18.224749,19.002001,18.50886,17.559082,17.934509,…,17.343284,17.32228,18.09385,18.24743,17.186804,18.499684,17.693258,18.077672,18.062232,17.713872,19.152533,19.131274,19.35725,19.792732,18.144107,18.847794,18.708151,17.466654,19.100675,18.646703,20.764949,18.407116,18.34649,17.201393,18.949739,17.815893,17.806289,18.41998,18.403305,18.612559,17.896223,17.811048,17.738377,18.072983,17.876114,19.132101,18.810775
"""Lipidomics""","""neg_LPG 22:6_[M-H]-__A""",15.939925,13.332694,16.258509,15.283197,15.376758,16.577574,15.052493,14.767001,14.222718,16.398119,14.580229,15.33611,15.582789,15.455396,16.647439,17.779687,15.101362,15.420531,13.334115,15.068705,11.487924,15.495652,15.474534,15.887301,16.103162,14.974481,16.0001,15.58053,13.274243,14.136811,15.689124,14.416469,16.326141,14.932304,15.580751,…,16.057472,15.539072,15.909858,16.348064,16.029266,16.033734,13.939998,15.704383,15.767129,15.287322,15.720675,17.551123,16.168995,14.918845,14.377889,16.884587,17.079708,14.215011,16.312236,14.498737,16.181298,15.295053,15.998223,14.699496,17.087171,15.056595,16.286271,15.399837,16.385283,15.616657,15.357581,14.53959,15.55933,16.920046,15.077261,15.648837,17.159722
"""Lipidomics""","""neg_LPI 20:3_[M-H]-__A""",17.03302,16.27659,15.580615,15.508878,16.293132,16.323765,17.241012,17.480965,17.282085,17.122331,16.118254,17.287977,16.577573,15.753691,16.950782,17.360527,15.800438,14.754988,16.349948,15.328972,15.710085,16.490614,15.084179,16.290234,17.52108,14.767972,16.073114,15.737933,16.196002,17.134708,15.495628,16.617408,14.670533,15.396166,16.425467,…,null,15.34704,15.368405,16.662644,14.96511,15.507413,14.400997,14.860501,15.805879,15.277685,15.539793,16.321945,17.502062,17.783228,14.919043,16.838498,16.257867,15.050656,16.020405,16.862748,19.834705,15.398115,16.207838,15.507567,16.793991,15.838628,15.612993,16.299807,15.78733,16.813625,16.083459,15.55696,16.03181,16.242012,15.510758,17.170929,16.818901
"""Lipidomics""","""neg_LPG 22:5_[M-H]-__A""",17.18434,13.769529,16.310356,16.559035,15.295408,16.291611,16.151017,14.927514,13.715198,17.136068,14.667662,15.932099,17.17814,16.371562,16.37376,15.805954,15.938042,15.261179,14.097894,15.129483,12.858515,14.915505,15.404731,15.983098,16.054884,15.875988,14.860855,14.881498,15.672194,14.906863,15.101946,15.46148,17.375415,15.131399,16.419864,…,17.129548,14.837918,17.063393,16.309982,15.903623,16.748122,15.327412,16.587611,16.912779,15.56203,17.112847,16.409388,16.175442,16.792172,15.327093,16.264481,17.435875,16.277012,17.675897,15.862881,16.742496,16.005721,16.926397,15.418413,16.67529,15.319853,16.338051,15.7259,16.169941,16.059122,16.186822,14.777519,15.323566,15.509279,

### Metabolomics

In [10]:
_ptrc, _pilot = import_metabolites(SYN)
metab = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Metabolomics").alias("Block"),
        pl.exclude("Block")
    )
)
metab

[syn71896311:rp_normalized_combat_edata_metab.csv]: Found existing file at /Users/dylan.ross/.synapseCache/125/166162125/rp_normalized_combat_edata_metab.csv, skipping download.
[syn25796769:PNNL_clinical_summary_03_03_2021.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/799/76897799/PNNL_clinical_summary_03_03_2021.xlsx, skipping download.
[syn68835814:PTRC_Exp26 Sample Key.xlsx]: Found existing file at /Users/dylan.ross/.synapseCache/432/161070432/PTRC_Exp26 Sample Key.xlsx, skipping download.


Block,Feature,12-00032,11-00503,12-00069,13-00468,18-00103,12-00123,18-00101,12-00127,12-00145,12-00154,12-00196,12-00250,12-00268,12-00294,18-00149,12-00383,13-00016,13-00033,13-00034,13-00047,13-00059,13-00075,13-00077,13-00092,13-00123,13-00157,13-00160,13-00186,13-00195,13-00262,13-00331,13-00393,13-00149,13-00147,13-00450,…,C-09-1074,C-09-1336,C-09-1608,C-09-1906,C-09-2182,C-09-3512,C-09-4769,C-09-5381,C-09-5462,C-10-0302,C-10-0535,C-10-0773,C-10-1211,C-10-3265,C-10-3429,C-10-3906,C-10-3924,C-11-0287,C-11-2295,C-11-5466,C-12-0858,C-12-1118,C-12-2943,C-12-4258,C-13-0276,16-00627,16-01191,16-00731,16-00494,17-00025,16-01100,17-00741,17-00881,16-00120,16-00292,16-01109,C-98-0665
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Metabolomics""","""rppos_3.Hydroxyhexadecanoylcar…",18.560512,20.456926,17.321128,21.317864,18.952151,17.618723,19.466776,18.478136,16.904964,19.840103,17.418109,18.516067,19.214826,20.425267,14.750338,18.797737,17.706429,15.090577,21.396441,19.2071,21.439233,15.3336,15.786503,17.069387,17.685282,18.325687,14.585494,14.723604,17.539325,19.83152,16.636314,16.612449,21.058499,17.125955,13.370101,…,17.346782,17.199605,18.997263,15.802918,16.928037,14.130853,16.533692,19.883155,15.123579,18.711861,18.370304,17.730964,16.343378,19.095459,14.035508,15.633157,17.096824,20.406942,19.191233,18.496756,16.844086,18.197451,20.079984,17.194435,16.975042,18.202975,18.060554,17.828468,19.83855,20.222337,19.482136,16.362991,20.37417,19.095295,18.580408,14.865952,22.366506
"""Metabolomics""","""rppos_5.Oxoproline""",23.962277,23.503443,23.409263,22.938007,23.494437,24.454826,24.910931,25.595707,22.867875,22.058986,22.631341,23.82986,23.228487,23.365667,24.593384,23.938365,25.438705,23.652126,23.412514,24.501844,23.098215,25.126152,24.079572,22.304367,24.382121,23.652235,22.984809,21.655368,24.959062,23.637517,23.975473,23.345713,22.216808,23.388193,24.333541,…,24.225233,23.570777,23.315419,24.741439,23.487553,24.163404,23.930632,23.773553,22.160474,25.474376,25.995752,23.040634,24.244641,23.610062,23.378467,21.990856,22.886282,25.55339,23.776649,24.145757,24.859096,24.926631,26.079559,22.005859,24.772171,23.549691,25.557336,25.353716,26.46191,25.568378,25.736419,25.673114,22.929829,24.907967,26.335303,23.470423,21.720762
"""Metabolomics""","""rppos_Acetyl.L.carnitine""",25.39362,24.277188,27.592381,29.240905,26.739821,23.035321,26.369587,26.426252,24.475636,22.735283,25.042691,26.015598,25.496206,27.549655,25.559441,26.534583,26.594596,25.540321,25.52127,27.29041,25.190277,26.635384,24.18679,24.010038,25.761064,24.878758,25.619571,26.68818,26.423349,23.455613,24.775523,26.847838,27.518276,23.247788,25.075348,…,25.992848,26.323137,25.159653,25.041541,24.604203,23.490563,25.435658,25.21754,23.14897,25.996345,25.144126,23.754881,27.10277,25.349666,24.383168,22.372676,24.943207,27.220559,25.474747,25.700509,25.59748,25.292423,26.220358,22.800699,25.930365,26.365187,28.034162,28.528256,29.156017,28.876271,28.880857,28.322491,27.146991,27.611863,29.263052,23.696274,23.24535
"""Metabolomics""","""rppos_Adenosine""",21.508335,22.36418,20.498261,24.566766,22.149919,20.014892,20.136825,21.626258,21.523075,18.798877,17.78252,23.661798,22.518133,23.221477,19.960924,21.876893,21.180283,23.805605,18.011,23.453698,22.977329,21.53343,19.882501,20.4344,22.455901,23.902172,22.004498,22.213514,21.371691,23.540279,22.020971,21.072412,23.364422,21.95915,20.890053,…,19.752544,19.351094,20.052811,20.54399,19.621588,19.873469,20.067605,19.730311,19.680589,21.79611,21.966103,17.864929,19.367325,19.302868,20.751488,17.502103,19.457855,21.466337,18.769726,20.654424,20.605302,20.587528,22.844771,20.251166,20.68268,21.096607,22.507803,23.227931,22.346316,22.748953,22.799608,22.102362,19.567887,25.151

### Proteomics

In [11]:
_ptrc, _pilot = import_global(SYN)
prot = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Proteomics").alias("Block"),
        pl.exclude("Block")
    )
)
prot

[syn73755181:global_beataml_batch_corrected.csv]: Found existing file at /Users/dylan.ross/Projects/AML_outcome_disparities/260422_prep_PR/amlMultidrugResistance/aml_outcome_disparities/src/python/data/global_beataml_batch_corrected.csv, skipping download.


[syn73755182:global_pilot_batch_corrected.csv]: Found existing file at /Users/dylan.ross/Projects/AML_outcome_disparities/260422_prep_PR/amlMultidrugResistance/aml_outcome_disparities/src/python/data/global_pilot_batch_corrected.csv, skipping download.


Block,Feature,11-00261,11-00503,11-00475,12-00032,11-00376,11-00378,11-00382,11-00388,11-00416,11-00465,11-00466,12-00069,12-00123,12-00127,12-00145,12-00196,13-00147,12-00294,12-00383,13-00033,13-00034,13-00075,13-00077,13-00123,13-00149,13-00157,13-00160,13-00186,13-00195,13-00226,13-00245,13-00262,13-00331,13-00393,13-00450,…,C-95-068,C-01-1665,C-09-1033,C-99-2136,C-05-1782,94-C-376,PS88-0050,C-01-2163,C-02-0350,C-04-0434,C-01-0171,PS88-0140,C-11-2295,C-99-1077,C-05-0004,C-05-3159,C-05-4372,PS89-0158,C-01-0930,C-98-0031,94-C-077,C-98-0033,C-98-0665,C-99-1700,C-02-1356,C-99-2065,14-00528-Bridge,16-00120-Bridge,16-00494-Bridge,16-00627-Bridge,16-00731-Bridge,16-01100-Bridge,16-01109-Bridge,16-01191-Bridge,17-00025-Bridge,17-00741-Bridge,17-00881-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Proteomics""","""A1BG""",1.010275,1.193825,-0.368686,1.793044,null,-1.084904,-1.341473,-0.94186,-2.553533,-0.831849,-1.4579,0.498278,null,null,1.063305,0.190083,-0.50073,-2.501175,0.434085,-1.00815,0.462745,-0.562803,0.418758,null,-0.983038,0.523952,1.824292,0.415318,1.214632,-0.334107,0.198923,1.641109,-0.930149,-0.042924,0.777459,…,0.503249,0.744407,-0.765565,0.070162,-0.685533,0.667158,0.552594,0.999923,1.321317,0.982233,1.120692,0.07014,-2.00289,1.131453,-0.045517,-0.963941,-0.375325,1.287504,0.757538,1.161624,2.295147,1.236394,0.016356,1.109179,2.191668,1.061017,-1.765944,-0.887179,-0.45448,-0.70102,-0.116854,-0.665767,-1.844439,0.771347,-0.732169,-0.548223,-0.349491
"""Proteomics""","""A2M""",0.609935,0.59216,0.141362,-0.051768,-0.343111,0.732146,-1.077609,-0.77956,-1.418408,-0.644383,-1.205371,-0.040282,1.350584,-1.198939,0.264238,-0.319335,-0.087844,-1.315691,0.664112,-0.462416,2.134339,0.236243,1.008567,2.131375,-1.116442,0.903392,2.607579,0.318913,1.21773,-1.271589,-0.548537,3.432149,-0.990183,1.489302,0.713376,…,1.333698,-0.180129,-1.292976,1.597939,-1.667052,0.91629,-0.032401,1.021053,1.063729,1.64441,1.247292,-0.119177,-2.654269,1.033691,-0.418087,-1.09193,-0.236866,1.105128,1.036175,1.550965,2.021506,1.930924,0.726604,1.640543,1.271923,1.004152,-0.644993,-1.482487,-0.357223,-0.401315,-0.884608,-0.142247,-1.297964,-0.170057,-0.63816,0.63802,0.221334
"""Proteomics""","""AAAS""",-0.607491,-0.31949,-1.156402,-1.084748,-1.621443,-2.679351,-1.890176,-2.105576,-2.097517,-1.562096,-1.579641,-0.707875,-0.973387,-1.795937,-0.683611,-1.15703,-1.408224,-2.741277,-0.561134,-1.456652,-0.516166,-1.54825,-1.058913,-0.300366,-2.115904,-0.439687,-1.462448,-1.501422,-0.879406,-3.128997,-2.450271,-0.502961,-1.456084,-1.519986,-0.367295,…,-1.64389,-2.614208,-1.141142,-1.529728,-2.248363,-2.002754,-1.404656,-0.401436,-1.29942,-0.406512,-1.511232,-1.187713,-1.705635,-0.750425,-0.701018,-1.391553,-0.809102,-1.349431,-0.49197,-1.284759,-1.156394,-0.782666,-2.167858,-0.816144,0.253734,-1.467816,-1.892983,-1.296235,-0.408886,-3.00823,-2.662839,-1.132003,-1.50009,-2.74196,-1.093548,-0.858019,-1.690165
"""Proteomics""","""AACS""",-0.176501,0.313655,-0.609557,-0.750311,-0.073949,-1.18306,-1.25451,-1.264321,-2.277627,-1.290747,-2.207366,-0.365852,-0.03859,-1.664651,-0.46153,-1.224686,-1.551044,-2.954592,-0.303504,-0.841197,0.788282,-1.329812,-0.078728,0.871793,-1.981292,-0.062389,-1.034471,-1.342467,0.222108,-3.033531,-2.152721,-0.491726,-1.520839,-1.66603,0.145144,…,-1.698733,-1.654561,-0.842278,-0.092,0.018038,-0.412975,-1.383157,-1.220393,0.266542,-0.240316,-1.964346,-0.944109,-2.00945,-1.581639,-0.445194,-0.923957,-0.569268,0.178282,-0.742533,-1.530399,-0.63022,-1.777491,-1.735419,-0.298714,-0.111104,-0.711214,-0.334524,-1.617261,0.869457,-0.951102,-1.529419,1.017977,-3.263164,-0.343664,0.061794,-0.270661,-0.613057
"""Proteomics""","""AAGAB""",-0.580698,-0.4766,-0.341627,-1.124299,-1.454701,-2.9

### Phosphoproteomics

In [12]:
_ptrc, _pilot = import_phospho(SYN)
phospho = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    _ptrc, 
                    include_index=True
                ),
                pl.from_pandas(
                    _pilot, 
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Phosphoproteomics").alias("Block"),
        pl.exclude("Block")
    )
)
phospho

[syn73754460:phospho_beataml_batch_corrected.csv]: Found existing file at /Users/dylan.ross/Projects/AML_outcome_disparities/260422_prep_PR/amlMultidrugResistance/aml_outcome_disparities/src/python/data/phospho_beataml_batch_corrected.csv, skipping download.


[syn73754465:phospho_pilot_batch_corrected.csv]: Found existing file at /Users/dylan.ross/Projects/AML_outcome_disparities/260422_prep_PR/amlMultidrugResistance/aml_outcome_disparities/src/python/data/phospho_pilot_batch_corrected.csv, skipping download.


Block,Feature,11-00261,11-00503,11-00475,12-00032,11-00376,11-00378,11-00382,11-00388,11-00416,11-00465,11-00466,12-00069,12-00123,12-00127,12-00145,12-00196,13-00147,12-00294,12-00383,13-00033,13-00034,13-00075,13-00077,13-00123,13-00149,13-00157,13-00160,13-00186,13-00195,13-00226,13-00245,13-00262,13-00331,13-00393,13-00450,…,C-95-068,C-01-1665,C-09-1033,C-99-2136,C-05-1782,94-C-376,PS88-0050,C-01-2163,C-02-0350,C-04-0434,C-01-0171,PS88-0140,C-11-2295,C-99-1077,C-05-0004,C-05-3159,C-05-4372,PS89-0158,C-01-0930,C-98-0031,94-C-077,C-98-0033,C-98-0665,C-99-1700,C-02-1356,C-99-2065,14-00528-Bridge,16-00120-Bridge,16-00494-Bridge,16-00627-Bridge,16-00731-Bridge,16-01100-Bridge,16-01109-Bridge,16-01191-Bridge,17-00025-Bridge,17-00741-Bridge,17-00881-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Phosphoproteomics""","""AAAS-S495s""",-0.228,-2.62,-0.111,null,null,null,-5.3,-3.58,-1.22,-1.25,-2.61,-5.23,null,0.0147,-0.892,-0.333,null,-1.59,null,null,-0.799,-0.17,-2.65,-1.5,-1.52,-4.13,-5.43,-3.17,-4.64,null,null,-0.259,-0.475,null,-0.534,…,-0.070358,-2.546159,-0.678956,-1.377696,-0.980433,-0.965089,0.253306,-0.104865,-0.410867,0.727484,-0.412822,-0.190408,-0.619896,-0.740228,0.130107,-1.06414,0.048576,0.037059,0.244974,-1.270726,-0.237886,0.814452,-2.727012,-0.16619,0.748594,-1.585919,-0.049049,-0.859636,0.357416,-0.980801,-1.391123,-0.274277,-0.033924,-3.043648,0.196599,-0.252866,-2.054192
"""Phosphoproteomics""","""AAK1-S20s""",0.693856,0.408721,-0.94158,-0.424579,0.703566,-1.732005,-0.635543,-0.537547,null,-0.744995,null,0.106242,0.397976,-0.890643,-0.55065,-1.309272,-0.519009,-1.895894,-0.125291,-0.299182,0.696533,-0.607979,0.165682,1.623686,-0.875183,-0.07325,-0.519544,-0.712439,0.482483,-1.931273,null,0.004459,-1.139513,-1.161068,-0.810165,…,null,-1.258214,null,0.279889,-0.810108,0.486404,null,0.843728,null,-0.090613,0.303768,0.751027,null,-1.09226,null,-0.260491,0.312562,1.288768,null,-0.413065,null,1.216551,-1.40153,0.008384,null,0.430239,-0.010062,-0.509899,null,null,-0.113599,null,-1.787005,null,0.559486,-0.197032,-0.677475
"""Phosphoproteomics""","""AAK1-S637s""",0.043272,2.317306,-0.775356,-6.389542,0.280428,-1.847803,-1.414204,-4.939151,-4.214155,-0.183061,-2.664992,-2.2941,-0.759245,-1.912305,-1.433181,-0.993864,-3.062853,-2.583468,-2.854584,-2.508529,0.61582,-0.394834,-1.486149,2.31749,-2.784627,-1.440515,-5.81749,-1.328161,-0.192808,-3.606629,null,0.253963,-1.524541,-4.716482,-1.396878,…,-2.508596,-3.459761,-2.866562,1.359772,-2.477557,-0.716246,-1.142248,0.565292,-0.527545,-4.32048,0.077927,0.061208,-2.722415,-2.353024,0.990825,-1.126458,-2.095451,0.731432,0.54202,-1.365001,-1.495029,-2.731382,-3.686851,-1.365721,-3.381137,-1.083227,-0.922248,-0.206884,-0.056312,-2.943713,-3.887491,0.014164,-3.874337,-3.538961,-0.148168,-2.16566,-2.468972
"""Phosphoproteomics""","""AAK1-S670sT674tS678s""",null,0.095059,-0.5958,-0.741351,null,-3.017537,null,-1.305877,-2.450587,-1.844609,null,-0.374305,-1.376778,-2.542745,null,null,null,-3.307811,-0.866881,-2.77231,-0.391756,null,-0.886811,1.119391,null,null,null,-2.381676,-0.036482,null,null,-0.643511,null,-3.935296,null,…,-2.556301,-3.289489,-1.689746,-0.42027,-1.838956,-1.401806,-1.230694,0.141009,0.242613,-1.061358,-0.079802,-0.558943,-0.831081,-2.023619,1.396926,-0.582053,-1.694131,0.115586,-0.252434,-0.593775,-2.432905,0.369233,-3.92299,-1.416649,-4.480351,-0.603058,-1.313252,-0.790995,-0.412861,-2.769204,-2.648549,1.243047,-2.989417,-1.971776,0.181735,-1.709711,-1.761734
"""Phosphoproteomics""","""AAK1-S678s""",2.024118,1.650225,0.630539,0.371569,null,1.134727,1.320309,1.000382,-0.49035,1.169107,null,1.421212,null,0.255917,null,null,0.667989,-0.236031,2.501417,0.530456,3.25502,0.698188,0.504447,1.764667,0.423262,0.747749

### Transcriptomics

In [13]:
def rename_redundant_columns(df):
    cols = []
    count = defaultdict(int)
    for column in df.columns:
        if df.columns.tolist().count(column) > 1:
            cols.append(f"{column}__{count[column] + 1}")
            count[column] += 1
        else:
            cols.append(column)
    df.columns = cols
    return df

In [14]:
_ptrc, _pilot = import_rna(SYN)
rna = (
    (_df := 
        pl.concat(
            [
                pl.from_pandas(
                    # rename duplicated columns with suffixes __1, __2, etc.
                    rename_redundant_columns(_ptrc.loc[:, pd.notnull(_ptrc.columns)]), 
                    include_index=True
                ),
                pl.from_pandas(
                    # rename duplicated columns with suffixes __1, __2, etc.
                    rename_redundant_columns(_pilot.loc[:, pd.notnull(_pilot.columns)]),
                    include_index=True
                ),
            ],
            how="diagonal"
        )
    )
    .select(pl.exclude("None"))
    .transpose(
        include_header=True, 
        header_name="Feature", 
        column_names=_df["None"].to_list()
    )
    .select(
        pl.lit("Transcriptomics").alias("Block"),
        pl.exclude("Block")
    )
)
rna

[syn72665045:RNAseq_beataml_batch_corrected.csv]: Found existing file at /Users/dylan.ross/Projects/AML_outcome_disparities/260422_prep_PR/amlMultidrugResistance/aml_outcome_disparities/src/python/data/RNAseq_beataml_batch_corrected.csv, skipping download.


[syn72665034:RNAseq_pilot_batch_corrected.csv]: Found existing file at /Users/dylan.ross/Projects/AML_outcome_disparities/260422_prep_PR/amlMultidrugResistance/aml_outcome_disparities/src/python/data/RNAseq_pilot_batch_corrected.csv, skipping download.


Block,Feature,12-00023,12-00051,12-00066,12-00150,12-00211,12-00258,12-00294,12-00372,12-00423,12-00426,13-00007,13-00016,13-00028,13-00034,13-00098,13-00118,13-00123,13-00126,13-00138,13-00145,13-00146,13-00147,13-00149,13-00150,13-00157,13-00160,13-00163,13-00165,13-00166,13-00195,13-00202,13-00204,13-00226,13-00232,13-00245,…,C-98-1247,C-99-0027,C-99-0740,C-99-0901,C-99-1077,C-99-1483,C-99-1700,C-01-1599,C-04-0434,C-05-0004,C-05-0319,C-05-3159,C-08-2091,C-09-0143,C-09-3512,C-09-5462,C-98-0665,C-00-0552,C-01-2163,C-05-3084,C-10-0535,C-11-0287,93-C-201,94-C-376,C-07-2900,C-98-0033,C-98-0846,C-99-2065,C-01-1665,C-12-2943,C-07-2070,C-08-3493,C-09-4769,C-05-0927,C-05-0664,C-00-1828,C-98-0031
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Transcriptomics""","""TSPAN6""",0.103667,0.655698,0.043228,0.640222,0.0,0.0,0.0,0.426814,0.186135,0.098528,0.034152,0.009039,0.0,0.0,0.0,0.065036,0.0,0.248771,0.046092,0.071954,0.026397,0.476622,0.094709,0.666167,0.0,0.554093,0.022238,0.123708,0.100289,0.044207,0.0,0.0,0.013633,0.014323,0.014626,…,0.473336,0.047715,0.613317,0.0,0.024252,0.308512,0.313306,0.436378,0.0,0.24898,0.623015,0.0,0.0,0.213534,0.111568,0.098088,0.978771,0.089795,0.0,1.633932,1.372375,0.150952,0.212802,0.080451,0.0,0.0,0.199976,0.431386,1.874078,0.0,0.119139,0.0,0.152393,0.966417,0.0,0.616858,0.0
"""Transcriptomics""","""DPM1""",42.29404,39.500211,32.368781,41.012229,35.795653,43.961225,46.675261,34.829337,32.094907,30.881377,40.313754,49.829357,40.343402,49.21178,58.11838,36.590721,34.977446,37.678645,43.418829,37.319361,61.775617,58.158374,43.794673,46.949734,46.678722,47.183763,47.357669,39.0392,45.513523,43.274931,47.599634,56.159523,44.858854,48.802168,39.186167,…,20.729275,29.910108,46.580027,53.544088,58.559764,55.951353,53.448866,38.177396,36.201168,22.934931,48.530889,38.464199,75.334858,56.983594,64.763746,52.558648,25.012581,28.683895,21.738902,44.990144,52.927616,31.282473,68.464404,38.06527,58.591159,40.625177,73.839927,38.73673,49.061317,11.190824,48.492913,48.800392,48.025827,25.725908,58.2125,51.204308,63.632347
"""Transcriptomics""","""SCYL3""",4.637419,4.248138,2.323953,3.786035,9.688157,3.872861,10.5317,9.178264,4.036026,9.534408,4.617206,6.430385,5.9423,4.268427,3.808378,6.775143,3.797193,9.248575,2.374682,4.93417,2.028796,4.512554,5.827037,6.197326,9.966101,1.751436,3.528992,8.091567,10.992753,4.973174,6.203744,3.582717,4.804516,5.295818,5.73135,…,6.7151,1.767103,3.595723,2.91495,5.852485,3.777857,2.5889,2.913312,4.595423,2.677044,3.721495,5.67976,4.93701,3.659757,5.957962,13.021958,4.139365,3.540088,4.924827,9.440076,3.768067,8.295576,4.343075,5.922965,3.57551,7.318193,9.695617,7.067862,3.449133,1.793744,3.522736,1.510576,1.775082,4.300711,0.0,3.31625,2.340134
"""Transcriptomics""","""FIRRM""",2.649891,0.925816,4.31647,3.969407,4.997223,3.365889,7.942433,4.710107,4.282121,3.773782,7.912462,7.149577,5.157053,2.627649,9.807955,5.297111,3.492096,3.908165,6.745768,1.859602,3.910666,2.974521,11.033194,6.533552,3.345746,4.893192,7.525145,4.980445,3.215329,3.95257,4.162512,3.405527,5.511774,5.173834,5.297837,…,5.74585,1.237826,1.956892,3.450212,3.869816,3.775247,2.121432,4.196561,3.755548,3.877682,3.485164,3.901122,5.994098,6.689297,3.757184,12.219012,6.055658,12.284732,2.658728,2.88443,6.528912,8.103024,2.499883,3.642569,4.759356,6.930527,5.840409,15.896855,1.388375,1.671598,1.516219,5.896507,4.363685,6.149518,0.0,7.045231,2.405106
"""Transcriptomics""","""FGR""",432.520601,6.684593,484.078954,19.653174,54.109046,577.275253,143.415806,90.834152,102.956098,30.409833,7.775333,81.418472,31.740665,332.737982,4.490878,108.653163,608.53827,69.790149,197.994472,32.982845,1.621894,65.4142,30.432193,122.352239,187.635077,267.163233,248.613079,47.351336,23.143257,439.

## Combine the (-Omics Data) Tables
(metadata table stays separate with samples as rows, to enable easier subsetting of samples based on metadata)

In [15]:
combined = (
    pl.concat(
        [
            acetyl,
            lipid,
            metab,
            prot,
            phospho,
            rna
        ],
        how="diagonal",
    )
    # keep only the sample columns that are included in the filtered metadata table
    .select(
        ["Block", "Feature"] + meta["Sample"].to_list()
    )
)
combined

Block,Feature,11-00261,11-00503,11-00475,12-00032,11-00376,11-00378,11-00382,11-00388,11-00416,11-00465,11-00466,12-00123,12-00127,12-00145,12-00196,12-00294,12-00383,13-00033,13-00034,13-00075,13-00077,13-00123,13-00149,13-00157,13-00160,13-00186,13-00195,13-00226,13-00262,13-00331,13-00450,13-00468,14-00495,13-00558,13-00581,…,C-95-068,C-01-1665,C-09-1033,C-99-2136,C-05-1782,94-C-376,PS88-0050,C-01-2163,C-02-0350,C-04-0434,C-01-0171,PS88-0140,C-11-2295,C-99-1077,C-05-0004,C-05-3159,C-05-4372,PS89-0158,C-01-0930,C-98-0031,94-C-077,C-98-0033,C-98-0665,C-99-1700,C-02-1356,C-99-2065,14-00528-Bridge,16-00120-Bridge,16-00494-Bridge,16-00627-Bridge,16-00731-Bridge,16-01100-Bridge,16-01109-Bridge,16-01191-Bridge,17-00025-Bridge,17-00741-Bridge,17-00881-Bridge
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Acetylomics""","""ACAA2-K234k""",-2.79,-1.32,-0.07,-0.762,null,null,-1.88,-2.1,-1.38,1.17,-1.37,0.484,null,-0.946,0.496,null,null,0.669,-0.13,-0.141,0.371,null,2.2,0.564,null,0.0392,1.01,-0.326,-0.0122,0.681,null,null,2.34,-0.522,0.571,…,-0.740195,0.056017,1.525606,-0.970053,0.912385,0.415692,-0.3854,-1.527722,-0.465969,-0.417754,0.374158,-0.285991,1.689777,-0.286936,-1.355843,-1.783394,-0.108936,-1.092201,-0.704166,0.16671,-0.802912,0.24294,-1.08595,-0.37591,1.766804,-1.017518,-1.686547,-2.171659,-1.095024,-1.067707,-0.778423,-0.543555,0.740597,0.360673,0.027206,-0.568194,0.311076
"""Acetylomics""","""ACAD9-K202k""",-0.195,-0.357,-0.36,null,0.905,0.207,-0.834,-0.373,null,0.0433,-0.799,null,-2.44,0.438,-0.0222,-0.513,0.389,null,null,-0.544,0.155,-0.136,null,-0.471,-0.511,0.214,1.19,null,0.694,-1.3,0.189,null,2.15,1.17,0.523,…,-0.07763,null,0.570992,null,1.1831,-1.511869,-0.009541,null,-1.085725,null,0.674999,-0.25304,1.075163,null,-0.507267,null,1.627,-0.660512,-0.644078,null,-0.622194,null,1.45712,0.155145,-1.235691,null,null,-0.566252,-0.724165,0.207464,-0.395339,0.362696,0.286919,-0.602037,0.194048,null,null
"""Acetylomics""","""ACADM-K304k""",-0.129,-0.0232,0.367,null,0.35,null,-0.53,0.457,-1.13,-1.24,1.59,0.114,-0.0785,-1.07,0.286,null,1.26,null,-0.313,0.366,-0.803,-1.18,1.79,0.585,-1.46,-0.663,1.05,1.96,-1.03,0.729,0.421,null,1.39,null,null,…,-0.025817,null,-0.310245,-1.19684,0.517396,0.369655,-0.694533,null,-2.428373,-0.117239,0.130378,-0.284755,1.54642,null,-1.205137,-1.247686,0.675267,-1.064369,-0.076896,null,-1.371136,0.38916,-1.102276,-1.272929,0.014274,null,null,-2.467522,-0.71158,-0.672011,-1.563158,-1.060444,0.034066,-0.994443,-0.024539,-0.595167,null
"""Acetylomics""","""ACADVL-K278k""",0.272,0.81,0.265,null,1.43,null,null,null,null,0.658,0.563,1.16,0.726,-1.03,1.4,-0.328,-0.582,-2.29,0.596,1.13,null,-0.225,1.37,-0.82,null,-1.46,0.175,0.0792,0.507,0.52,-0.926,null,-0.152,null,-1.26,…,-0.416003,-0.432192,1.182928,-1.049746,-1.909679,-0.075547,-0.960016,-0.708188,-1.025387,-0.443936,1.004101,-0.314048,1.280458,0.231344,-0.711069,-0.104481,-0.064754,-1.169472,0.946654,-1.107658,-0.412209,1.016264,-2.307984,-0.028166,0.831407,-2.584908,0.002312,-1.849316,-0.621441,-2.109005,-1.389221,-0.992909,0.247213,-1.049815,0.351727,-0.419173,-1.870117
"""Acetylomics""","""ACADVL-K298k""",-0.622,0.421,-0.478,-2.19,1.26,null,null,2.19,-0.656,0.235,0.0553,0.699,-0.605,0.465,0.915,null,1.16,null,null,0.149,0.188,0.265,0.865,-0.456,0.31,-0.647,1.38,null,-0.979,null,-1.5600e-25,null,0.974,-0.542,-1.03,…,-1.109983,-0.410377,0.883659,-0.129818,-0.651763,0.020461,0.199043,-0.233644,-0.517838,-0.023063,0.561925,0.312539,0.861988,0.109789,-0.905637,0.285302,-0.125828,-0.331017,0.154637,0.459963,-0.478133,0.02304,-0.588967,-0.360287,-0.094358,-0.797413,0.303983,0.212966,-1.009953,-0.655982,-0.293991,-0.178373,-0.209228,-0.471836,-0.166835,-1.296326,-0.359604
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

## Cache the Assembled Tables

In [18]:
meta.write_ipc(META_CACHE)

In [19]:
meta.write_csv(META_CACHE_CSV)

In [20]:
combined.write_ipc(COMBINED_CACHE)

In [21]:
combined.write_csv(COMBINED_CACHE_CSV)

## Upload assembled tables to synapse
I think this should only be done once to initialize the entries on synapse. After this, I think future updates should use the synapse IDs to upload updated copies?

In [22]:
syn_meta_ipc = scFile(
    path=META_CACHE,
    parent_id=SYN_UPLOAD_FOLDER_ID
)
syn_meta_ipc.store()
print(syn_meta_ipc.id)

SynapseHTTPError: 403 Client Error: insufficient_scope. Request lacks scope(s) required by this service: modify

Based on the above error, there seems to be a permission issue. I manually uploaded the files I am trying to cache and will come back and fix this code later to automatically update these files on synapse, which should not need to happen very often.

Synapse IDs:
- `metadata.csv`: syn74775940
- `metadata.arrow`: syn74775944
- `combined_omics_data.csv`: syn74775999
- `combined_omics_data.arrow`: syn74775996